# Setting Up the Environment

In [85]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

original_data = pd.read_csv('./dataset1.csv', parse_dates=True) # reading data from csv file
data = original_data.copy()  # making a copy so we don't modify the original dataset
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 907 entries, 0 to 906
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   start_time                 907 non-null    object 
 1   bat_landing_to_food        907 non-null    float64
 2   habit                      866 non-null    object 
 3   rat_period_start           907 non-null    object 
 4   rat_period_end             907 non-null    object 
 5   seconds_after_rat_arrival  907 non-null    int64  
 6   risk                       907 non-null    int64  
 7   reward                     907 non-null    int64  
 8   month                      907 non-null    int64  
 9   sunset_time                907 non-null    object 
 10  hours_after_sunset         907 non-null    float64
 11  season                     907 non-null    int64  
dtypes: float64(2), int64(5), object(5)
memory usage: 85.2+ KB


### Checking for any null value

In [86]:
data.isna().sum()

start_time                    0
bat_landing_to_food           0
habit                        41
rat_period_start              0
rat_period_end                0
seconds_after_rat_arrival     0
risk                          0
reward                        0
month                         0
sunset_time                   0
hours_after_sunset            0
season                        0
dtype: int64

# Analysing Habit Column
When we checked for null value in our dataset1. We found Habit column contains 41 null value so we are analyzing the data from habit column

In [87]:
data['habit'].isna().sum()  # calcualting the number of row that contains NaN value.
data['habit'].unique()  # analyzing the uniqueness of data in habit column
data['habit'].value_counts()


habit
fast                                                245
rat                                                 221
pick                                                139
bat                                                  30
bat_fight                                            26
                                                   ... 
bat_fight_and_rat                                     1
rat_and_rat                                           1
not_sure_rat                                          1
501.0,358.4,636.2,423.4; 476.0,103.0,634.0,206.0      1
rat_and_bat_and_pick                                  1
Name: count, Length: 81, dtype: int64

### Cleaning Habit Column
After analyzing habit column, we knew it contains junk data and some strings contains characters like 'rat_attack'. At this stage, we don't know how important habit column. So we will be replacing the junk data and NaN value with `Unknown` string.

In [88]:
def clean_habit(row):
    if pd.isna(row):    #checking if given value is null value or not
        return 'Unknown'
    val_check = str(row).strip()  # converting everything to strings
    
    if re.fullmatch(r'^[\d\.,;\s]+$', val_check): # checks if strings is made of digit, dots or character
        return 'Unknown'
    
    val = val_check.lower().replace('_', ' ')  # lower strings and replacing '_' with white space
    return re.sub(r'\s+', ' ', val).strip()

data.loc[:,'habit'] = data['habit'].map(clean_habit)  # assigning cleaned column to copied df


In [89]:
data['habit'].value_counts()

habit
fast                    245
rat                     221
pick                    139
Unknown                  58
bat                      30
                       ... 
fight bat                 1
bat fight and rat         1
rat and rat               1
not sure rat              1
rat and bat and pick      1
Name: count, Length: 64, dtype: int64

### Grouping data with less frequency 
After removing the junk, we can see there are few data with less frequency. We decided to grouping them into an 'other' category instead of removing them. This approach prevents data loss while simplifying the dataset.

In [90]:
def categorize_low_frequency(col, threshold: int):
    counts = col.value_counts() # calculate the frequency of each unique string in the series.
    low_freq = counts[counts < threshold].index.tolist() # identify values with a frequency less than given thresold
    cleaned_data = data['habit'].replace(low_freq, 'Other')
    return cleaned_data

cat_habit =  categorize_low_frequency(data['habit'], 3)
data.loc[:, 'habit'] = cat_habit

# Converting to datetime
We have some columns that contain date and time as data, however, their dtype is object. We are trying to conver them into datetime. 

In [91]:
cat_cols = ['start_time', 'rat_period_start', 'rat_period_end', 'sunset_time'] #colmns that contain data in {date time} format.
for col in cat_cols:  # Converting dtype object in to dtype datetime.
    data[col] = pd.to_datetime(data[col], dayfirst = True)

# Creating new Column called Date
data['Date'] = data['start_time'].dt.date

# Converting T.D. in Second and Hour into same Unit.
Few columns contains the time difference between two events in second, where as other columns contains the time differnce in hour. For our analyis, we are converting them into same unit i.e Minute.

In [92]:
# Converting different value like seconds or hours in to minute
data['bat_landing_to_food'] = data['bat_landing_to_food'] / 60
data['hours_after_sunset'] = data['hours_after_sunset'] * 60
data['seconds_after_rat_arrival'] = data['seconds_after_rat_arrival'] / 60


# We are chaning the column name suitable for new value 
data = data.rename(columns={'bat_landing_to_food' : 'bat_landing_to_food_min',
                            'hours_after_sunset' : 'min_after_sunset',
                            'seconds_after_rat_arrival' : 'min_after_rat_arrival'})

data.head()

,start_time,bat_landing_to_food_min,habit,rat_period_start,rat_period_end,min_after_rat_arrival,risk,reward,month,sunset_time,min_after_sunset,season,Date
0,2017-12-30 18:37:00,0.266667,rat,2017-12-30 18:35:00,2017-12-30 18:38:00,1.800000,1,0,0,2017-12-30 16:45:00,112.250000,0,2017-12-30
1,2017-12-30 19:51:00,0.001234,fast,2017-12-30 19:50:00,2017-12-30 19:55:00,0.283333,0,1,0,2017-12-30 16:45:00,186.050000,0,2017-12-30
2,2017-12-30 19:51:00,0.066667,fast,2017-12-30 19:50:00,2017-12-30 19:55:00,0.683333,0,1,0,2017-12-30 16:45:00,186.450000,0,2017-12-30
3,2017-12-30 19:52:00,0.166667,rat,2017-12-30 19:50:00,2017-12-30 19:55:00,1.850000,1,0,0,2017-12-30 16:45:00,187.616667,0,2017-12-30
4,2017-12-30 19:54:00,0.250000,rat,2017-12-30 19:50:00,2017-12-30 19:55:00,3.233333,1,0,0,2017-12-30 16:45:00,189.000000,0,2017-12-30


### Performing Chronological check

In [93]:
data['chronology_check'] = (data['rat_period_start'] <= data['start_time']) & (data['start_time'] <= data['rat_period_end'])


### Checking for any duplicates

In [94]:
data.duplicated().sum() # checking for duplicate
data = data.drop_duplicates() # droping the duplicate rows


# Calculating and Resolving Outlier

### Function to detect outlier

In [95]:
def detect_outlier(col):
   num = pd.to_numeric(col, errors='coerce') # Making sure all the values are numeric.
   q1 = num.quantile(0.25) # calculating first quantile
   q3 = num.quantile(0.75) # calcualting third quantile
   IQR = q3 - q1 # calculating Interquartile range
   lo = q1 - (1.5 * IQR) # calculating lower fence for data, a potential outlier.
   hi = q3 + (1.5 * IQR) # calculating high fence for data, a potential outlier
   outlier = (num < lo) | (num > hi)
   return outlier  # returns boolen value


In [96]:
check = detect_outlier(data['bat_landing_to_food_min'])



### Resolving outlier from given column

We are replacing the outlier with median value. Because, if we remove the outlier then it might create an data imbalance. 

In [97]:
#Selecting the columns to be check for outlier
col_name = ['bat_landing_to_food_min', 'min_after_sunset', 'min_after_rat_arrival'] 
for col in col_name:
    outlier = detect_outlier(data[col])  # detecting the outliers row in a given col
    median_value = data[col].median() # calculating median of given value
    data.loc[outlier, col] = median_value # flagging specific row for given column

check = detect_outlier(data['bat_landing_to_food_min'])


# Saving the Cleaned Data to a New Dataset

In [99]:
data.to_csv("dataset1_cleaned.csv", index=False)